# Assistant Médical Medi-Info
> **Outil d'aide pour pharmaciens** — Vérification des interactions médicamenteuses

**Technologies utilisées :**
- `LangChain` — framework agent
- `OpenAI` — modèles GPT (sélection dynamique)
- `FAISS` + `SentenceTransformers` — recherche sémantique RAG
- `python-dotenv` — gestion de la clé API

##  Étape 1 — Installation des dépendances

In [1]:
!pip install langchain langchain-openai openai faiss-cpu sentence-transformers python-dotenv numpy>=2.0.0


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


##  Étape 2 — Chargement de la clé API

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("Clé API chargée avec succès")
else:
    print("Clé API introuvable — vérifiez votre fichier .env")

✅ Clé API chargée avec succès


##  Étape 3 — Création des documents médicaux
Les notices médicales sont sauvegardées dans le dossier `documents/`

In [ ]:
import os

os.makedirs("documents", exist_ok=True)

notices = {
    "documents/paracetamol.txt": """Médicament : Paracétamol (aussi appelé doliprane, efferalgan, dafalgan)
Classe : Analgésique et antipyrétique
Indications : Traitement des douleurs légères à modérées, fièvre
Dose maximale adulte : 4000 mg par jour, soit 1g toutes les 6 heures
Dose pédiatrique : 15 mg par kg de poids corporel

Contre-indications :
- Insuffisance hépatique sévère
- Allergie au paracétamol

Interactions dangereuses :
- Warfarine (anticoagulant) : DANGER, augmente l'effet anticoagulant, risque hémorragique
- Alcool : RISQUE, toxicité hépatique fortement accrue, éviter toute consommation""",

    "documents/ibuprofene.txt": """Médicament : Ibuprofène (aussi appelé advil, nurofen, brufen)
Classe : Anti-inflammatoire non stéroïdien (AINS)
Indications : Douleurs, fièvre, inflammations, rhumatismes
Dose maximale adulte : 1200 mg par jour
Dose pédiatrique : 10 mg par kg de poids corporel

Contre-indications :
- Ulcère gastrique ou intestinal actif
- Insuffisance rénale sévère
- Grossesse au 3ème trimestre

Interactions dangereuses :
- Warfarine : DANGER, risque hémorragique très accru
- Aspirine : RISQUE, augmentation des effets indésirables gastro-intestinaux
- Lithium : DANGER, augmente fortement la toxicité du lithium""",

    "documents/warfarine.txt": """Médicament : Warfarine (aussi appelé coumadine, anticoagulant oral)
Classe : Anticoagulant antivitamine K
Indications : Prévention des thromboses veineuses, embolie pulmonaire, fibrillation auriculaire

Contre-indications :
- Grossesse
- Saignement actif ou risque hémorragique élevé

Interactions dangereuses :
- Paracétamol : DANGER, potentialise l'effet anticoagulant
- Ibuprofène : DANGER, risque hémorragique majeur
- Aspirine : DANGER, risque hémorragique très élevé
- Amoxicilline : RISQUE, déséquilibre de l'INR""",

    "documents/amoxicilline_aspirine.txt": """Médicament : Amoxicilline (clamoxyl, augmentin)
Classe : Antibiotique pénicilline
Indications : Infections bactériennes ORL, pulmonaires, urinaires
Dose pédiatrique : 25 mg par kg de poids corporel

Contre-indications :
- Allergie aux pénicillines

---

Médicament : Aspirine (acide acétylsalicylique, aspégic, kardégic)
Classe : Analgésique, antiagrégant plaquettaire
Indications : Douleurs, fièvre, prévention cardiovasculaire

Contre-indications :
- Ulcère gastrique actif
- Enfants de moins de 16 ans en cas de fièvre virale

Interactions dangereuses :
- Warfarine : DANGER, risque hémorragique très élevé
- Ibuprofène : RISQUE, réduit l'effet cardioprotecteur"""
}

for chemin, contenu in notices.items():
    with open(chemin, "w", encoding="utf-8") as f:
        f.write(contenu)

print(f" {len(notices)} documents médicaux créés dans documents/")

✅ 4 documents médicaux créés dans documents/


##  Étape 4 — Moteur RAG (Recherche Sémantique)
Vectorisation des documents avec **FAISS** + **SentenceTransformers**

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

class RAGEngine:
    def __init__(self, documents_dir="documents"):
        print("[RAG] Chargement du modèle d'embeddings...")
        self.model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
        self.documents = []
        self.metadatas = []
        self.index = None
        self._charger_documents(documents_dir)
        self._construire_index()
        print(f"[RAG] {len(self.documents)} chunks indexés.")

    def _charger_documents(self, dossier):
        for fichier in os.listdir(dossier):
            if fichier.endswith(".txt"):
                with open(os.path.join(dossier, fichier), "r", encoding="utf-8") as f:
                    contenu = f.read()
                chunks = [c.strip() for c in contenu.split("\n\n") if c.strip()]
                for chunk in chunks:
                    self.documents.append(chunk)
                    self.metadatas.append(fichier)

    def _construire_index(self):
        embeddings = self.model.encode(self.documents, show_progress_bar=True)
        embeddings = np.array(embeddings, dtype="float32")
        faiss.normalize_L2(embeddings)
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)

    def rechercher(self, question, top_k=3):
        vecteur = self.model.encode([question], show_progress_bar=False)
        vecteur = np.array(vecteur, dtype="float32")
        faiss.normalize_L2(vecteur)
        scores, indices = self.index.search(vecteur, top_k)
        resultats = []
        for score, idx in zip(scores[0], indices[0]):
            if idx != -1 and score > 0.3:
                source = self.metadatas[idx].replace(".txt", "")
                resultats.append(f"[Source: {source}]\n{self.documents[idx]}")
        return "\n\n---\n\n".join(resultats) if resultats else "Aucun document pertinent trouvé."

rag = RAGEngine()
print("\n Test RAG avec 'doliprane' :")
print(rag.rechercher("doliprane", top_k=1))

C:\Users\admin\scoop\apps\python\current\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[RAG] Chargement du modèle d'embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]

[RAG] ✅ 15 chunks indexés.

🔎 Test RAG avec 'doliprane' :
[Source: amoxicilline_aspirine]
---


##  Étape 5 — Les deux outils LangChain

In [ ]:
from langchain.tools import tool

@tool
def query_medical_database(drug_name: str) -> str:
    """
    Recherche les informations d'un médicament dans la base de données médicale
    via recherche sémantique RAG. Fonctionne avec les noms de marque (doliprane, advil...)
    """
    print(f"[RAG] Recherche pour : '{drug_name}'")
    return rag.rechercher(drug_name, top_k=3)

@tool
def convert_dosage(weight: float, dose_mg: float) -> str:
    """
    Calcule la dose précise pour un patient selon son poids.
    - weight  : poids du patient en kg
    - dose_mg : dose recommandée en mg/kg
    """
    if weight <= 0 or dose_mg <= 0:
        return "Erreur : poids et dose doivent être supérieurs à 0."
    dose_totale = weight * dose_mg
    return (
        f"Calcul de dosage :\n"
        f"  Poids          : {weight} kg\n"
        f"  Dose/kg        : {dose_mg} mg/kg\n"
        f"  Dose totale    : {dose_totale:.1f} mg"
    )

tools = [query_medical_database, convert_dosage]
print(" Outils LangChain créés :")
for t in tools:
    print(f"  - {t.name} : {t.description[:60]}...")

✅ Outils LangChain créés :
  - query_medical_database : Recherche les informations d'un médicament dans la base de d...
  - convert_dosage : Calcule la dose précise pour un patient selon son poids.
- w...


##  Étape 6 — Sélection dynamique de modèle + Création de l'agent

| Mot-clé détecté | Modèle utilisé |
|---|---|
| interaction, danger, contre-indication, risque | `gpt-4-turbo` |
| Autres questions | `gpt-3.5-turbo` |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

KEYWORDS_DANGER = ["interaction", "danger", "contre-indication", "risque"]

def choisir_modele(question: str) -> str:
    question_lower = question.lower()
    for mot in KEYWORDS_DANGER:
        if mot in question_lower:
            print(f"⚠️  Mot-clé '{mot}' détecté → gpt-4-turbo")
            return "gpt-4-turbo"
    print(" Question simple → gpt-3.5-turbo")
    return "gpt-3.5-turbo"

def creer_agent(modele: str) -> AgentExecutor:
    llm = ChatOpenAI(model=modele, temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Tu es Medi-Info, un assistant médical professionnel pour pharmaciens.
Tu réponds uniquement aux questions liées aux médicaments.
Tu utilises les outils disponibles pour rechercher les fiches médicales et calculer les doses.
Sois précis, clair et prudent. Rappelle toujours de consulter un médecin pour les cas sensibles."""),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    agent = create_tool_calling_agent(llm, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

def repondre(question: str) -> str:
    modele = choisir_modele(question)
    agent = creer_agent(modele)
    return agent.invoke({"input": question})["output"]

print("✅ Agent Medi-Info prêt !")

✅ Agent Medi-Info prêt !


##  Étape 7 — Tests de l'agent
### Test 1 : Question simple (gpt-3.5-turbo attendu)

In [7]:
from IPython.display import Markdown

reponse = repondre("Quelles sont les informations sur le doliprane ?")
Markdown(reponse)

💬 Question simple → gpt-3.5-turbo


> Entering new AgentExecutor chain...

Invoking: `query_medical_database` with `{'drug_name': 'doliprane'}`


[RAG] Recherche pour : 'doliprane'
[Source: amoxicilline_aspirine]
---Je n'ai pas pu trouver d'informations spécifiques sur le Doliprane dans la base de données médicales. Voulez-vous que je fasse une autre recherche ou que je vous fournisse des informations générales sur le paracétamol (principe actif du Doliprane) ?

> Finished chain.


Je n'ai pas pu trouver d'informations spécifiques sur le Doliprane dans la base de données médicales. Voulez-vous que je fasse une autre recherche ou que je vous fournisse des informations générales sur le paracétamol (principe actif du Doliprane) ?

### Test 2 : Question avec mot-clé danger (gpt-4-turbo attendu)

In [8]:
reponse = repondre("Y a-t-il un danger entre l'ibuprofène et la warfarine ?")
Markdown(reponse)

⚠️  Mot-clé 'danger' détecté → gpt-4-turbo


> Entering new AgentExecutor chain...

Invoking: `query_medical_database` with `{'drug_name': 'ibuprofène'}`


[RAG] Recherche pour : 'ibuprofène'
[Source: ibuprofene]
Médicament : Ibuprofène (aussi appelé advil, nurofen, brufen)
Classe : Anti-inflammatoire non stéroïdien (AINS)
Indications : Douleurs, fièvre, inflammations, rhumatismes
Dose maximale adulte : 1200 mg par jour
Dose pédiatrique : 10 mg par kg de poids corporel

---

[Source: amoxicilline_aspirine]
Interactions dangereuses :
- Warfarine : DANGER, risque hémorragique très élevé
- Ibuprofène : RISQUE, réduit l'effet cardioprotecteur

---

[Source: amoxicilline_aspirine]
Médicament : Aspirine (acide acétylsalicylique, aspégic, kardégic)
Classe : Analgésique, antiagrégant plaquettaire
Indications : Douleurs, fièvre, prévention cardiovasculaire
Invoking: `query_medical_database` with `{'drug_name': 'warfarine'}`


[RAG] Recherche pour : 'warfarine'
[Source: warfarine]
Médicament : W

Il existe une interaction dangereuse entre l'ibuprofène et la warfarine. L'utilisation concomitante de ces deux médicaments peut augmenter le risque de saignements en raison de l'effet anticoagulant de la warfarine et des propriétés anti-inflammatoires de l'ibuprofène qui peuvent affecter la coagulation sanguine. Il est donc recommandé de consulter un médecin avant de combiner ces médicaments pour évaluer les risques et ajuster les doses si nécessaire.

### Test 3 : Calcul de dosage

In [9]:
reponse = repondre("Calcule la dose d'amoxicilline pour un enfant de 30 kg")
Markdown(reponse)

💬 Question simple → gpt-3.5-turbo


> Entering new AgentExecutor chain...

Invoking: `convert_dosage` with `{'weight': 30, 'dose_mg': 25}`


Calcul de dosage :
  Poids          : 30.0 kg
  Dose/kg        : 25.0 mg/kg
  Dose totale    : 750.0 mgLa dose d'amoxicilline recommandée pour un enfant de 30 kg est de 750 mg.

> Finished chain.


La dose d'amoxicilline recommandée pour un enfant de 30 kg est de 750 mg.

## Étape 8 — Interface interactive
Posez vos propres questions à l'agent Medi-Info

In [10]:
question = "Quelles sont les contre-indications de l'aspirine ?"  # ← modifiez ici

reponse = repondre(question)
Markdown(reponse)

⚠️  Mot-clé 'contre-indication' détecté → gpt-4-turbo


> Entering new AgentExecutor chain...

Invoking: `query_medical_database` with `{'drug_name': 'aspirine'}`


[RAG] Recherche pour : 'aspirine'
[Source: amoxicilline_aspirine]
Médicament : Aspirine (acide acétylsalicylique, aspégic, kardégic)
Classe : Analgésique, antiagrégant plaquettaire
Indications : Douleurs, fièvre, prévention cardiovasculaire

---

[Source: warfarine]
Médicament : Warfarine (aussi appelé coumadine, anticoagulant oral)
Classe : Anticoagulant antivitamine K
Indications : Prévention des thromboses veineuses, embolie pulmonaire, fibrillation auriculaire

---

[Source: ibuprofene]
Interactions dangereuses :
- Warfarine : DANGER, risque hémorragique très accru
- Aspirine : RISQUE, augmentation des effets indésirables gastro-intestinaux
- Lithium : DANGER, augmente fortement la toxicité du lithiumL'aspirine, ou acide acétylsalicylique, a plusieurs contre-indications importantes à prendre en compte :

1. **Allergies 

L'aspirine, ou acide acétylsalicylique, a plusieurs contre-indications importantes à prendre en compte :

1. **Allergies à l'aspirine ou aux salicylates** : Les personnes ayant des antécédents d'allergie à l'aspirine ou à d'autres salicylates ne doivent pas prendre ce médicament.

2. **Troubles de la coagulation** : L'aspirine ayant un effet antiagrégant plaquettaire, elle est contre-indiquée chez les personnes souffrant de troubles de la coagulation ou chez celles qui prennent déjà des anticoagulants, car elle peut augmenter le risque de saignements.

3. **Ulcères gastriques ou duodénaux actifs** : L'aspirine peut irriter la muqueuse gastrique et aggraver ou provoquer des ulcères.

4. **Insuffisance rénale ou hépatique sévère** : L'aspirine peut aggraver ces conditions en raison de son métabolisme et de son excrétion qui impliquent le foie et les reins.

5. **Asthme induit par les salicylates** : Chez certaines personnes, l'aspirine peut déclencher des crises d'asthme ou d'autres réactions allergiques.

6. **Enfants et adolescents avec des infections virales** : L'utilisation de l'aspirine est associée au syndrome de Reye, une maladie rare mais grave, chez les enfants et adolescents qui ont des infections virales, notamment la varicelle et la grippe.

Il est crucial de consulter un professionnel de santé avant de commencer un traitement à l'aspirine pour s'assurer qu'il n'y a pas de contre-indications spécifiques à votre situation.